# MayaScan Model Training

Train the U-Net archaeological feature detection model on the **Chactún ML-ready dataset**.

This notebook is designed to run on **Google Colab with a free T4 GPU**.

**Dataset:** [Kokalj et al., 2023](https://doi.org/10.1038/s41597-023-02455-x) — 2,094 tiles, 130 km², 10K+ annotated Maya structures.

**Classes:** Buildings (9,303), Platforms (2,110), Aguadas/reservoirs (95)

In [ ]:
# Install dependencies
!pip install -q segmentation-models-pytorch torch torchvision tqdm Pillow
!pip install -q huggingface-hub  # for model upload

In [ ]:
import os
import zipfile
import urllib.request

DATA_DIR = "/content/chactun_data" if os.path.exists("/content") else "./chactun_data"
os.makedirs(DATA_DIR, exist_ok=True)

# Chactún ML-ready dataset from Figshare
# https://figshare.com/articles/dataset/22202395
FIGSHARE_URL = "https://figshare.com/ndownloader/articles/22202395/versions/3"
ZIP_PATH = os.path.join(DATA_DIR, "chactun.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "extracted")

if not os.path.exists(EXTRACT_DIR):
    print("Downloading Chactún dataset from Figshare...")
    urllib.request.urlretrieve(FIGSHARE_URL, ZIP_PATH)
    print(f"Downloaded to {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}")
else:
    print("Dataset already downloaded.")

# Show what we got
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = " " * 2 * (level + 1)
        for f in files[:5]:
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image


class ChactunDataset(Dataset):
    """Load Chactún tiles: 3-channel viz input + 3 binary masks → multi-class."""

    def __init__(self, data_dir, split="train", augment=True):
        self.data_dir = data_dir
        self.augment = augment and (split == "train")

        # Find all SVF tiles (use as index for matching other channels)
        svf_pattern = os.path.join(data_dir, "**", "*svf*.*")
        all_svf = sorted(glob.glob(svf_pattern, recursive=True))
        # Filter to image files only
        all_svf = [f for f in all_svf if f.lower().endswith((".tif", ".tiff", ".png"))]

        if len(all_svf) == 0:
            # Try case-insensitive search
            svf_pattern = os.path.join(data_dir, "**", "*SVF*.*")
            all_svf = sorted(glob.glob(svf_pattern, recursive=True))
            all_svf = [f for f in all_svf if f.lower().endswith((".tif", ".tiff", ".png"))]

        print(f"Found {len(all_svf)} SVF tiles")

        # 80/20 split
        n = len(all_svf)
        split_idx = int(n * 0.8)
        if split == "train":
            self.svf_files = all_svf[:split_idx]
        else:
            self.svf_files = all_svf[split_idx:]

        print(f"{split}: {len(self.svf_files)} tiles")

    def _find_matching(self, svf_path, keyword):
        """Find matching openness/slope/mask file for a given SVF file."""
        directory = os.path.dirname(svf_path)
        basename = os.path.basename(svf_path)
        # Replace 'svf'/'SVF' with target keyword
        for old in ["svf", "SVF", "Svf"]:
            target = basename.replace(old, keyword)
            if target != basename:
                break
        candidate = os.path.join(directory, target)
        if os.path.exists(candidate):
            return candidate
        # Fallback: search in same directory
        for f in os.listdir(directory):
            if keyword.lower() in f.lower() and f.lower().endswith((".tif", ".tiff", ".png")):
                return os.path.join(directory, f)
        return None

    def _load_image(self, path):
        """Load image as float32 numpy array."""
        img = Image.open(path)
        return np.array(img, dtype=np.float32)

    def _normalize(self, arr):
        """Normalize to [0, 1]."""
        vmin, vmax = arr.min(), arr.max()
        if vmax - vmin > 1e-10:
            return (arr - vmin) / (vmax - vmin)
        return np.zeros_like(arr)

    def __len__(self):
        return len(self.svf_files)

    def __getitem__(self, idx):
        svf_path = self.svf_files[idx]

        # Load 3 visualization channels
        svf = self._normalize(self._load_image(svf_path))
        opns_path = self._find_matching(svf_path, "opns")
        slope_path = self._find_matching(svf_path, "slope")

        opns = self._normalize(self._load_image(opns_path)) if opns_path else np.zeros_like(svf)
        slope = self._normalize(self._load_image(slope_path)) if slope_path else np.zeros_like(svf)

        # Stack to (3, H, W)
        image = np.stack([svf, opns, slope], axis=0).astype(np.float32)

        # Load masks → combine into multi-class target
        mask = np.zeros(svf.shape[:2] if svf.ndim > 1 else svf.shape, dtype=np.int64)
        for cls_id, keyword in [(1, "building"), (2, "platform"), (3, "aguada")]:
            mask_path = self._find_matching(svf_path, keyword)
            if mask_path and os.path.exists(mask_path):
                m = self._load_image(mask_path)
                if m.ndim > 2:
                    m = m[:, :, 0]  # take first channel
                mask[m > 0.5] = cls_id

        # Augmentation (rotation + flip — direction-independent visualizations)
        if self.augment:
            k = np.random.randint(4)
            image = np.rot90(image, k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k).copy()
            if np.random.rand() > 0.5:
                image = np.flip(image, axis=2).copy()
                mask = np.flip(mask, axis=1).copy()

        return torch.from_numpy(image), torch.from_numpy(mask)


# Test the dataset
ds = ChactunDataset(EXTRACT_DIR, split="train")
if len(ds) > 0:
    img, msk = ds[0]
    print(f"Image shape: {img.shape}, Mask shape: {msk.shape}")
    print(f"Mask classes present: {torch.unique(msk).tolist()}")
else:
    print("No tiles found — check the dataset structure above and adjust file patterns.")

In [ ]:
import segmentation_models_pytorch as smp
from torch import nn, optim
from tqdm import tqdm

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

# Model
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=4,
).to(device)

# Class weights (inverse frequency: background common, aguadas very rare)
class_weights = torch.tensor([0.1, 1.0, 2.0, 50.0], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Data
train_ds = ChactunDataset(EXTRACT_DIR, split="train", augment=True)
val_ds = ChactunDataset(EXTRACT_DIR, split="val", augment=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

EPOCHS = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Training: {len(train_ds)} tiles, Validation: {len(val_ds)} tiles")
print(f"Batch size: 8, Epochs: {EPOCHS}")

In [ ]:
# Training loop
best_iou = 0.0

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_loss = train_loss / max(len(train_dl), 1)

    # Validate
    model.eval()
    iou_sum = 0.0
    iou_count = 0
    with torch.no_grad():
        for images, masks in val_dl:
            images, masks = images.to(device), masks.to(device)
            preds = model(images).argmax(dim=1)
            # Mean IoU over non-background classes
            for c in range(1, 4):
                pred_c = preds == c
                mask_c = masks == c
                intersection = (pred_c & mask_c).sum().float()
                union = (pred_c | mask_c).sum().float()
                if union > 0:
                    iou_sum += (intersection / union).item()
                    iou_count += 1

    mean_iou = iou_sum / max(iou_count, 1)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, val_mIoU={mean_iou:.4f}, lr={scheduler.get_last_lr()[0]:.6f}")

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.state_dict(), "mayascan_unet_best.pth")
        print(f"  ✓ Saved best model (mIoU={best_iou:.4f})")

print(f"\nTraining complete! Best validation mIoU: {best_iou:.4f}")

In [ ]:
# Upload trained model to Hugging Face Hub
from huggingface_hub import HfApi, login

# Login (will prompt for token in Colab)
login()

api = HfApi()

# Create repo if it doesn't exist
REPO_ID = "fascinated23/mayascan"  # Change to your HF username
try:
    api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
except Exception as e:
    print(f"Repo may already exist: {e}")

# Upload model weights
api.upload_file(
    path_or_fileobj="mayascan_unet_best.pth",
    path_in_repo="mayascan_unet_v1.pth",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Model uploaded to https://huggingface.co/{REPO_ID}")

## Next Steps

After training, the model can be used with MayaScan:

```python
import mayascan

# Download trained weights
from huggingface_hub import hf_hub_download
model_path = hf_hub_download("fascinated23/mayascan", "mayascan_unet_v1.pth")

# Run detection with trained model
result = mayascan.process_dem(dem, model_path=model_path)
```

Or launch the web app:
```bash
python app.py
```